# Feature Selection

Uses only the **dev segment** (default: first 20% of each project), preceding each CV fold to prevent data leakage.

The resulting feature list and size are frozen and shared by GP and flow-control solutions.

Target: **T2** (first failures), on **atlasdb** project only.

In [ ]:
from dataclasses import replace

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import matthews_corrcoef
from sklearn.model_selection import TimeSeriesSplit

from acise_p1_cigp.analysis import features as fs
from acise_p1_cigp.data.load import PASS_LABEL, Dataset, load_project, preprocess
from acise_p1_cigp.viz import plots

PROJECT = "atlasdb"
SEED = 42
MIN_FREQ = 0.6
RHO = 0.8


def dev(ds: Dataset):
    return ds.X[ds.dev_mask].reset_index(drop=True), ds.y[ds.dev_mask].reset_index(drop=True)


raw = load_project(PROJECT)
dt1, dt2 = preprocess(raw, "T1"), preprocess(raw, "T2")

X1, y1 = dev(dt1)
X2, y2 = dev(dt2)

display(
    pd.DataFrame(
        {
            tag: {"dev_rows": len(y), "dev_failed": int(y.sum()), "fail_%": round(y.mean() * 100, 4)}
            for tag, y in (("T1", y1), ("T2", y2))
        }
    )
)

## Clean

Filter methods: missing-value ration and variance threshold.

- **Nominal codes** droppede, because a GP `lt/ge` threshold on unordered codes is meaningless.
- **Missing > 30%** dropped; the rest is median imputed (`line_added` has around 8% missing from the `-1` sentinel).
- **Near-constant**: `VarianceThreshold(threshold=1e-3)` on min-max scaled data.
- **Rare binary feature**: minority value < 1%; a 1% minority column still has variance around 0.01, so the threshold above misses it.

In [ ]:
def reason(c: str) -> str:
    if c in fs.NOMINAL:
        return "nominal"
    if X2[c].isna().mean() > 0.3:
        return "missing"
    return "rare binary" if c in binary else "near-constant"


Xc = fs.clean(X2)
dropped = sorted(set(X2.columns) - set(Xc.columns))
binary = fs._binary(X2)

print(f"{X2.shape[1]} -> {Xc.shape[1]} features after cleaning ({len(dropped)} dropped)")
display(pd.Series({c: reason(c) for c in dropped}, name="reason").to_frame())

## Relevance

The statistic depends on the feature type:

1. Numeric: ANOVA F, Kendall tau, mutual information (MI)
2. Binary: chi-square, mutual information = information gain.

Churn and count columns are heavily skewed, so rank- and information-based measures are used, not Pearson. `score` is the mean **percentile** rank over the statistics that apply, which keeps numeric and binary feature comparable.

In [ ]:
scores = fs.relevance(Xc, y2, seed=SEED)
display(scores.head(25).round(4))

In [ ]:
ranks = scores.drop(columns="score").rank(pct=True)
fig = plots.scores_bar(ranks, top=25)
plots.save(fig.update_layout(title="Relevance percentile rank per statistic (T2, top 25)"), "feature_relevance_T2")
fig.show()

# MI also capture threshold effects that the monotone statistics do not capture;
# so a feature that ranks high only on MI points to a threshold rule, exactly how we will model if-then rules.

## Redundancy

mRMR-style feature walking; keeping features that have their |Spearman rho| <= 0.8 with respect to ALL kept features.

Near-duplicate pairs are removed, and only one feature from that family is kept.

In [ ]:
ranking = scores.index.tolist()
kept = fs.prune_redundant(Xc, ranking, rho=RHO)
print(f"{len(ranking)} -> {len(kept)} features after redundancy pruning (rho={RHO})")

fig = plots.corr_heatmap(Xc[ranking[:30]].corr(method="spearman"))
plots.save(fig.update_layout(height=800, title="Spearman correlation heatmap (T2, top 30)"), "feature_corr_T2")
fig.show()

In [ ]:
rep = fs._family_rep(Xc, kept, rho=RHO)

fam = pd.Series(rep, name="representative").to_frame()
fam = fam[fam.index != fam.representative]
fam["abs_spearman"] = [abs(Xc[f].corr(Xc[r], method="spearman")) for f, r in fam.representative.items()]

display(fam.sort_values(["representative", "abs_spearman"], ascending=[True, False]).round(4))

### Data notes (RavenBuild source, `get_open_source_features.py`)

- **Copied columns.** `class_changed` and `met_sig_modified` are filled with `import_deleted_line`; all three are equal on every row (rho = 1). The computed class-rename value is discarded and cannot be recovered without the GumTree diffs. Whichever of the three is kept (here `met_sig_modified`) means **`import_deleted`**.
- **Inverted names.** In RavenBuild, `result == 1` means *pass*. `fail_ratio_re`, `fail_ratio_com_*` and `consec_fail_builds_*` therefore measure passes. `fail_ratio_pr` (`1 - mean`) and `file_fail_prob_*` (`result == 0`) do measure failures.
- **No leakage in the history features.** They are computed only from builds strictly before the current one (`iloc[:truncate_idx]`).
- `gh_team_size` falls into the `fail_ratio_pr_inc` family (ρ ≈ 0.89). Both drift with project age.

## Stability and threshold

Clean, relevance, and redundancy are repeated on 20 **block subsamples** of dev: 80% of contiguous blocks (length sqrt3(n)) are drawn without replacment and kept in time order.

A feature is selected in a subsample if it is in the top `n_max=20` after pruning. The **frozen list** is the pruned features with frequency >= `MIN_FREQ`, in relevance order.

In [ ]:
sel = fs.select_features(dt2, min_freq=MIN_FREQ, rho=RHO, seed=SEED)
print("N = ", sel.size)

display(
    pd.DataFrame({"stability": sel.stability[sel.features], "score": sel.scores.loc[sel.features, "score"]}).round(4)
)

In [ ]:
fig = plots.stability_bar(sel.stability[sel.stability > 0])
fig.add_hline(y=MIN_FREQ, line_dash="dash", annotation_text=f"min_freq={MIN_FREQ}", annotation_position="bottom right")
plots.save(fig.update_layout(title="Feature selection stability (T2)"), "feature_stability_T2")
fig.show()

In [ ]:
THRESHOLDS: list[float] = [0.5, 0.6, 0.75, 0.9, 0.95, 0.99]


def frozen(t: float, n_max: int = 20) -> list[str]:
    return [f for f in kept if sel.stability.get(f, 0) >= t][:n_max]


assert frozen(MIN_FREQ) == sel.features, "frozen list should match selected features"
display(
    pd.DataFrame(
        [{"min_freq": t, "N": len(frozen(t)), "features": ", ".join(frozen(t))} for t in THRESHOLDS]
    ).set_index("min_freq")
)

## Embedded Check

Used a **Random Forest** classifier with `balanced` class weights and parameters:

- `n_estimators`: 300
- `class_weight`: "balanced"
- `random_state`: SEED

to estimate the feature importance according to permutations in the weak classifiers from the ensemble. The RF is trained on 70% of the dev segment, while ROC AUC (permutation importance) is computed on the remaining 30%.

Features in the forest's top N that the filters dropped are recorded, and the frozen feature list does not get mutated.

In [ ]:
imp = sel.importance.head(20).rename("importance").rename_axis("feature").reset_index()
imp["selected"] = imp.feature.isin(sel.features)

fig = px.bar(imp, x="feature", y="importance", color="selected")
plots.save(fig.update_layout(title="Feature importance (T2, top 20)"), "feature_importance_T2")
fig.show()

display(
    pd.DataFrame(
        {
            "importance": sel.importance[sel.disagreements],
            "stability": sel.stability.reindex(sel.disagreements),
            "family": [rep.get(f, f) for f in sel.disagreements],
        }
    ).round(4)
)

## Drift

The dev slice stays at the start of the project, which keeps it leakage-free.
To see how the choice holds up over time, selection is re-run on each CV's fold train (dev + expanding window).

A frozen feature counts as kept if it or a near-duplicate is in that fold's top 20.

Diagnostic, the frozen feature list is not mutated.

In [ ]:
dev_idx = np.flatnonzero(dt2.dev_mask)
rest = np.flatnonzero(~dt2.dev_mask)
trains = [np.r_[dev_idx, rest[tr]] for tr, _ in TimeSeriesSplit(n_splits=5).split(rest)]

dr = fs.drift(dt2, trains, sel.features, rho=RHO, seed=SEED)
fig = plots.drift_heatmap(dr)
plots.save(fig.update_layout(title="Feature selection drift (T2)"), "feature_drift_T2")
fig.show()

display(dr.mean().round(4).rename("share kept").to_frame().T)
print("lost in >= 3 folds: ", dr.index[(~dr).sum(axis=1) >= 3].tolist())

In [ ]:
DROP = dr.index[(~dr).sum(axis=1) >= 3].tolist()
sel = replace(sel, features=[f for f in sel.features if f not in DROP], meta = sel.meta | {"dropped_due_to_drift": DROP})
print(f"After dropping {len(DROP)} features due to drift, N = ", sel.size)

## T1 Analysis

We expect `last_label` to dominate during selection as it is the most tied feature to a CI's status.

In [ ]:
sel1 = fs.select_features(dt1, min_freq=MIN_FREQ, rho=RHO, seed=SEED)
sel1_nl = fs.select_features(replace(dt1, X=dt1.X.drop(columns="last_label")), min_freq=MIN_FREQ, rho=RHO, seed=SEED)

display(
    pd.DataFrame(
        {
            "with_last_label": pd.Series(sel1.features),
            "without_last_label": pd.Series(sel1_nl.features),
        }
    )
)

print("last_label - rank:", sel1.scores.index.get_loc("last_label") + 1, "| stability: ", sel1.stability["last_label"])
print("Parrot MCC (dev):", round(matthews_corrcoef(y1, (X1.last_label != PASS_LABEL).astype(int)), 4))

**Result:** the expectation does not hold on atlasdb.

- `last_label` is a candidate in every subsample (only `sel1_nl` removes it), yet it ranks 30th on relevance and its stability is 0: it never reaches the top 20.
- Parrot (`prediction = last_label`) reaches MCC 0.075 on dev.
- The lists with and without `last_label` share 16 of 17 features; only `fail_ratio_pr_inc` vs `prev_srcchurn` differ in the tail.

## Per-project Check

The same selection is run on T2 for other candidates projects, selected by the amount of diversity in the build statuses (pass vs. failure).

Their dev slices might be small and contain few first failures, so their lists are noisy and purely indicative.

In [ ]:
projs = ["gradle-baseline", "cassandra", "rskj", "spring-cloud-aws"]
sels = {PROJECT: sel} | {
    p: fs.select_features(preprocess(load_project(p), "T2"), min_freq=MIN_FREQ, rho=RHO, seed=SEED) for p in projs
}

display(
    pd.DataFrame(
        {
            p: {
                "dev_rows": s.meta["dev_rows"],
                "dev_failed": s.meta["dev_failed"],
                "fail_%": round(s.meta["dev_failed"] / s.meta["dev_rows"] * 100, 4),
                "N": s.size,
            }
        for p, s in sels.items()
        }
    ).T
)

In [ ]:
def jaccard(a: list[str], b: list[str]) -> float:
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if a | b else np.nan


names = list(sels)
jac = pd.DataFrame(
    [[jaccard(sels[a].features, sels[b].features) for b in names] for a in names],
    index=names,
    columns=names,
)

fig = px.imshow(jac.round(4), text_auto=True, zmin=0, zmax=1, aspect="auto", color_continuous_scale=plots.COLOR_SCALE)
plots.save(fig.update_layout(title="Jaccard similarity of frozen feature lists (T2)"), "feature_overlap_T2")
fig.show()

## Freeze

In [ ]:
for s in (sel, sel1):
    print(s.save(), "| N =", s.size)